### Execução do Fluxo de ELT

In [2]:
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName('etl_pipeline') \
    .config("spark.jars", "/opt/spark/jars/iceberg-spark-runtime-3.5_2.12-1.6.0.jar") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.spark_catalog.type", "hive") \
    .config("spark.sql.catalog.local.warehouse", "s3a://datalake/iceberg") \
    .getOrCreate()

#Ajuste de log WARN log para ERROR
spark.sparkContext.setLogLevel("ERROR")

In [3]:
## Instalar Bibliotecas
!pip install python-dotenv==1.0.1

In [4]:
import os
from dotenv import load_dotenv, dotenv_values

import json
import requests

from pyspark.sql import DataFrame
from pyspark.sql.functions import explode, col, from_utc_timestamp
from pyspark.sql.types import StructType, StructField, StringType, ArrayType

In [5]:
%run ../common/Common_env_functions.ipynb

In [7]:
%run ../common/Common_data_functions.ipynb


### Carregar Variavel de Ambiente

In [9]:
# list_env_var('../.env')
load_dotenv('../.env')
token = os.getenv("API_TOKEN")

### Endpoints da API 

In [10]:
API_URL = 'https://api.olhovivo.sptrans.com.br/v2.1'
AUT = f"Login/Autenticar?token={token}"
LINHA = "Linha/Buscar?termosBusca="
L_POSICAO = "Posicao/Linha?codigoLinha="
L_PARADA = "Parada/BuscarParadasPorLinha?codigoLinha="
L_PREVISAO = "Previsao/Linha?codigoLinha="
EMPRESA = "Empresa"
POSICAO = 'Posicao'

id_linha = '8000'

### Schema da Tabela

In [11]:
schema = [
    StructField("c", StringType(), True),
    StructField("cl", StringType(), True),
    StructField("sl", StringType(), True),
    StructField("lt0", StringType(), True),
    StructField("lt1", StringType(), True),
    StructField("qv", StringType(), True),
    StructField("vs", ArrayType(
        StructType([
            StructField("p", StringType(), True),
            StructField("a", StringType(), True),
            StructField("ta", StringType(), True),
            StructField("py", StringType(), True),
            StructField("px", StringType(), True),
            StructField("sv", StringType(), True),
            StructField("is", StringType(), True),
        ])
    ), True)
]

## Extrair os Dados

In [12]:
# Orquestração Ingestão

try:
    auth_url = f"{API_URL}/{AUT}"

    session, authentication = auth(auth_url)

    if authentication:
        end_point = f"{API_URL}/{POSICAO}"

        data = fetch_data(session, authentication, end_point)
        
        if len(data):
            df_olhovivo = create_dataframe(schema, data['l'])
    
except Exception as e:
    print(f"Something went wrong: {e}") 

In [13]:
df_olhovivo.printSchema()

root
 |-- c: string (nullable = true)
 |-- cl: string (nullable = true)
 |-- sl: string (nullable = true)
 |-- lt0: string (nullable = true)
 |-- lt1: string (nullable = true)
 |-- qv: string (nullable = true)
 |-- vs: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- p: string (nullable = true)
 |    |    |-- a: string (nullable = true)
 |    |    |-- ta: string (nullable = true)
 |    |    |-- py: string (nullable = true)
 |    |    |-- px: string (nullable = true)
 |    |    |-- sv: string (nullable = true)
 |    |    |-- is: string (nullable = true)



## Escrever os dados no Data Lake

In [14]:
df_olhovivo.select("*").toPandas()

,c,cl,sl,lt0,lt1,qv,vs
0,5290-10,33365,2,TERM. PQ. D. PEDRO II,DIV. DIADEMA,5,"[(63351, true, 2025-04-26T22:51:53Z, -23.68085..."
1,606C-10,35363,2,CANTINHO DO CÉU,CIRCULAR,2,"[(66111, true, 2025-04-26T22:51:43Z, -23.74263..."
2,2704-10,916,1,METRÔ ITAQUERA,JD. ROBRU,3,"[(36685, true, 2025-04-26T22:52:03Z, -23.52665..."
3,6042-10,33987,2,STO. AMARO,JD. TRÊS ESTRELAS,2,"[(78361, true, 2025-04-26T22:52:02Z, -23.64408..."
4,978T-10,889,1,METRÔ BARRA FUNDA,JD. GUARANI,7,"[(16708, true, 2025-04-26T22:51:46Z, -23.52851..."
...,...,...,...,...,...,...,...
1874,701H-10,35326,2,CIRCULAR,HOSPITAL GUARAPIRANGA,1,"[(78840, true, 2025-04-26T22:51:44Z, -23.70528..."
1875,476L-10,33030,2,LAR ESC. SÃO FRANCISCO,METRÔ VL. MARIANA,1,"[(64304, true, 2025-04-26T22:52:02Z, -23.58954..."
1876,175P-10,33443,2,ANA ROSA,METRÔ SANTANA,7,"[(21784, true, 2025-04-26T22:51:57Z, -23.53466..."
1877,8013-43,2622,1,TERM. JD. BRITANIA,MORRO DOCE,1,"[(16112, true, 2025-04-26T22:52:03Z, -23.43001..."


In [15]:
spark.sql("USE iceberg")

DataFrame[]

In [16]:
(
    df_olhovivo
    .writeTo("iceberg.silver.tbl_silver_olhovivo")
    .createOrReplace()
)

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


In [17]:
spark.sql("SHOW TABLES in silver").show()

+---------+-------------------+-----------+
|namespace|          tableName|isTemporary|
+---------+-------------------+-----------+
|   silver|tbl_silver_olhovivo|      false|
+---------+-------------------+-----------+



## Transformar os Dados

In [18]:
df_olhovivo_gold = spark.sql("""
    SELECT
     c AS Letreiro_Linha,
     cl AS Linha,
     sl AS Sentido,
     lt0 AS Destino_Linha,
     lt1 AS Origem_Linha,
     CAST(qv AS INT) AS Quantidade_Veiculos,
     vs
     
    FROM iceberg.silver.tbl_silver_olhovivo"""
).withColumn(
    "vs",
    explode(col("vs"))    
).select(
    "Letreiro_Linha",
    "Linha",
    "Sentido",
    "Destino_Linha",
    "Origem_Linha",
    "Quantidade_Veiculos",
    col("vs.p").alias("Prefixo_Veiculo"),
    col("vs.a").cast('boolean').alias("Veiculo_Acessivel"),
    from_utc_timestamp(col("vs.ta"),"America/Sao_Paulo").alias("Horario"),
    col("vs.py").alias("Latitude"),
    col("vs.px").alias("Longitude")   
    
)

In [19]:
df_olhovivo_gold.printSchema()

root
 |-- Letreiro_Linha: string (nullable = true)
 |-- Linha: string (nullable = true)
 |-- Sentido: string (nullable = true)
 |-- Destino_Linha: string (nullable = true)
 |-- Origem_Linha: string (nullable = true)
 |-- Quantidade_Veiculos: integer (nullable = true)
 |-- Prefixo_Veiculo: string (nullable = true)
 |-- Veiculo_Acessivel: boolean (nullable = true)
 |-- Horario: timestamp (nullable = true)
 |-- Latitude: string (nullable = true)
 |-- Longitude : string (nullable = true)



In [20]:
(
    df_olhovivo_gold
    .writeTo("iceberg.gold.tbl_gold_olhovivo")
    .createOrReplace()
)

In [22]:
spark.sql("select * from iceberg.gold.tbl_gold_olhovivo limit 10; ").toPandas()

,Letreiro_Linha,Linha,Sentido,Destino_Linha,Origem_Linha,Quantidade_Veiculos,Prefixo_Veiculo,Veiculo_Acessivel,Horario,Latitude,Longitude
0,5290-10,33365,2,TERM. PQ. D. PEDRO II,DIV. DIADEMA,5,63351,True,2025-04-26 19:51:53,-23.680852,-46.636013000000005
1,5290-10,33365,2,TERM. PQ. D. PEDRO II,DIV. DIADEMA,5,63290,True,2025-04-26 19:52:00,-23.6811735,-46.636431
2,5290-10,33365,2,TERM. PQ. D. PEDRO II,DIV. DIADEMA,5,63368,True,2025-04-26 19:51:37,-23.575192,-46.640798000000004
3,5290-10,33365,2,TERM. PQ. D. PEDRO II,DIV. DIADEMA,5,63375,True,2025-04-26 19:52:04,-23.64941375,-46.640634750000004
4,5290-10,33365,2,TERM. PQ. D. PEDRO II,DIV. DIADEMA,5,63247,True,2025-04-26 19:51:33,-23.61655275,-46.63886600000001
5,606C-10,35363,2,CANTINHO DO CÉU,CIRCULAR,2,66111,True,2025-04-26 19:51:43,-23.742632999999998,-46.658328499999996
6,606C-10,35363,2,CANTINHO DO CÉU,CIRCULAR,2,66112,True,2025-04-26 19:51:58,-23.7406495,-46.6570705
7,2704-10,916,1,METRÔ ITAQUERA,JD. ROBRU,3,36685,True,2025-04-26 19:52:03,-23.526659000000002,-46.409887999999995
8,2704-10,916,1,METRÔ ITAQUERA,JD. ROBRU,3,36021,True,2025-04-26 19:51:44,-23.521501,-46.442832499999994
9,2704-10,916,1,METRÔ ITAQUERA,JD. ROBRU,3,36016,True,2025-04-26 19:51:57,-23.5236605,-46.418086


### Manutenção

In [ ]:
# spark.sql("SHOW TABLES in silver").show()

In [23]:
spark.sql("SHOW TABLES in gold").show()

+---------+-----------------+-----------+
|namespace|        tableName|isTemporary|
+---------+-----------------+-----------+
|     gold|tbl_gold_olhovivo|      false|
+---------+-----------------+-----------+



In [ ]:
## Para deletar por completo do catalog e storage
# spark.sql("DROP TABLE iceberg.bronze.tbl PURGE")

In [ ]:
# spark.stop()